# What?
Survey of file structure and contents created in notebooks 00-03. Used PySpark for experience. 
# Why?
A whole bunch of files and directories were created in the previous notebooks. They are stored in your Colab server, which doesn't have a convenient GUI to visualize what just happened. This should help with that. 
# Where?
Connect to the same Colab server you connected to in notebooks 00-03

## Initialize Spark Session and Confirm Mount
Setup

In [1]:
from pyspark.sql import SparkSession
import os

# Initialize spark session with local master configuration
spark = SparkSession.builder \
    .appName("colab_file_survey") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Spark Version: {spark.version}")
print(f"Spark Context is running with {sc.appName}")

Spark Version: 4.0.5
Spark Context is running with colab_file_survey


Locate and verify Google Drive mount path

In [2]:
#Should be at /content/drive
base_dir = "/content"#/drive/MyDrive"
# target_folder = "/Cadenza Challenge"

if os.path.exists(base_dir):
    print(f"Drive path verified: {base_dir}")
else:
    print(f"Warning: {base_dir} not found. Check your mount and ensure you have run notebook 00")

Drive path verified: /content


## Walk Directory and Explore File Types

In [3]:
from pathlib import Path

def collect_file_metadata(root_path):
    file_records = []
    root_p = Path(root_path)

    for path in root_p.rglob("*"):
        if path.is_file():
            try:
                stat = path.stat()
                parts = path.relative_to(root_p).parts
                top_level_dir = parts[0] if len(parts) > 1 else 'root_level'

                file_records.append({
                    "file_path":str(path),
                    "file_name":path.name,
                    "top_directory":top_level_dir,
                    "extension":path.suffix.lower() if path.suffix else "no_extension",
                    "size_mb": round(stat.st_size / (1024 * 1024), 4),
                    "date_modified":stat.st_mtime,
                    # "parent_dir":str(path.parent)
                })
            except Exception as e:
                continue
    return file_records

#Collect metadata locally, then parellalize into a Spark RDD for analysis (not strictly necessary, for leanring)
records = collect_file_metadata(base_dir)
file_df = spark.createDataFrame(records)
file_df.cache()

# Summary
print(f"Total files discovered: {file_df.count()}")


Total files discovered: 3428


In [4]:
from pyspark.sql.functions import col, from_unixtime, desc
from datetime import datetime, timedelta

date_you_ran_00_03 = "2026-11-12"

base_date = datetime.strptime(str(date_you_ran_00_03), "%Y-%m-%d")
end_date = base_date + timedelta(days=1)

date_configured_start = base_date.strftime("%Y-%m-%d 00:00:00")
date_configured_end = end_date.strftime("%Y-%m-%d 00:00:00")

print("Structure:\n")
recent_files_df = file_df.withColumn("timestamp", from_unixtime(col('date_modified'))).filter((col("timestamp") >= date_configured_start) & (col("timestamp") <= date_configured_end))
print(f"Total files discovered: {recent_files_df.count()}")
recent_files_df.groupBy("top_directory").count().show(50, truncate=False)

cadenza_dir = [row.top_directory for row in recent_files_df.select("top_directory").distinct().collect()]
print(cadenza_dir)


Structure:

Total files discovered: 2847
+---------------+-----+
|top_directory  |count|
+---------------+-----+
|clarity        |657  |
|exp_one_scene  |5    |
|cadenza_data   |1640 |
|zenodo_download|539  |
|root_level     |1    |
|outputs        |4    |
|drive          |1    |
+---------------+-----+

['clarity', 'exp_one_scene', 'cadenza_data', 'zenodo_download', 'root_level', 'outputs', 'drive']


In [ ]:
#write over this list if something is missing:
cadenza_dir = ['clarity', 'exp_one_scene', 'cadenza_data', 'zenodo_download', 'root_level', 'outputs', 'drive']
important_files_df = recent_files_df.filter(col("top_directory").isin(cadenza_dir))``

In [6]:
# 1. Collect the file paths from the Spark DataFrame into a local Python list
file_paths = [row.file_path for row in important_files_df.select("file_path").collect()]

# 2. Build a nested dictionary representing the file tree
tree = {}
for path in file_paths:
    # Split the path into parts (ignoring empty strings from leading/trailing slashes)
    parts = [p for p in path.split('/') if p]
    current = tree
    for part in parts:
        current = current.setdefault(part, {})

# 3. Define a recursive function to print the tree visually
def print_tree(node, prefix=""):
    pointers = [192, 186, 196] # └── and ├── characters
    # Get sorted keys so files/folders are displayed alphabetically
    keys = sorted(node.keys())
    
    for i, key in enumerate(keys):
        is_last = (i == len(keys) - 1)
        connector = "└── " if is_last else "├── "
        print(prefix + connector + key)
        
        # If the node has children (i.e., it's a directory), recurse deeper
        if node[key]:
            extension = "    " if is_last else "│   "
            print_tree(node[key], prefix + extension)

# 4. Print the generated file tree
print("Visual File Tree Structure:")
print_tree(tree)

Visual File Tree Structure:
└── content
    ├── cadenza_cad2_task2_eval.v1_1.tar.gz
    ├── cadenza_data
    │   └── cad2
    │       └── task2
    │           ├── audio
    │           │   ├── train
    │           │   │   ├── 02-lamortdase-strings
    │           │   │   │   └── Mix_1
    │           │   │   │       ├── Oboe.flac
    │           │   │   │       ├── Viola.flac
    │           │   │   │       ├── Violin_1.flac
    │           │   │   │       ├── Violin_2.flac
    │           │   │   │       └── mix_02-lamortdase-strings.flac
    │           │   │   ├── AgitatoDaFiereTempeste
    │           │   │   │   └── Mix_1
    │           │   │   │       ├── Cello.flac
    │           │   │   │       ├── Viola.flac
    │           │   │   │       ├── Violin_1.flac
    │           │   │   │       ├── Violin_2.flac
    │           │   │   │       └── mix_AgitatoDaFiereTempeste.flac
    │           │   │   ├── AlTardarDellaVendetta
    │           │   │   │   └── Mix_1
    │        

In [57]:
spark.stop()